# LeetCode 643 — Maximum Average Subarray I bằng Python

**Link đề:** https://leetcode.com/problems/maximum-average-subarray-i/
**Độ khó:** Easy · **Chủ đề:** Array, Sliding Window

```python
class Solution:
    def findMaxAverage(self, nums: list[int], k: int) -> float:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **subarray liên tiếp** (contiguous subarray) có độ dài cố định `k`.
2. Viết được cách brute force bằng slicing `nums[i:i+k]` và thấy vì sao nó chậm (`O(n·k)`).
3. Nắm kỹ thuật **cửa sổ trượt cố định** (fixed-size sliding window): thêm phần tử mới, bỏ phần tử cũ.
4. Biết thêm cách **prefix sum** (tổng tiền tố) như một cách nhìn khác cùng `O(n)`.
5. Hiểu mẹo "so sánh tổng thay vì so sánh trung bình" vì `k` cố định, chỉ chia **một lần** ở cuối.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO không tự chạy.

## 1. Đề bài — ta cần tìm gì?

Cho mảng số nguyên `nums` có `n` phần tử và số nguyên `k`.
Tìm **một đoạn liên tiếp dài đúng `k`** có **trung bình lớn nhất**, trả về giá trị trung bình đó (sai số < `10^-5` được chấp nhận).

**Ví dụ 1:** `nums = [1,12,-5,-6,50,3]`, `k = 4` → `12.75`

Các đoạn dài 4:

| Vị trí bắt đầu `i` | Đoạn `nums[i:i+4]` | Tổng | Trung bình |
|---:|---|---:|---:|
| 0 | `[1, 12, -5, -6]` | 2 | 0.5 |
| 1 | `[12, -5, -6, 50]` | 51 | **12.75** |
| 2 | `[-5, -6, 50, 3]` | 42 | 10.5 |

**Ví dụ 2:** `nums = [5]`, `k = 1` → `5.0`

**Constraints:**
- `n == len(nums)`, `1 <= k <= n <= 10^5`
- `-10^4 <= nums[i] <= 10^4`

Câu hỏi then chốt: có `n - k + 1` đoạn; tính tổng mỗi đoạn từ đầu thì tốn `k` phép cộng. Có cách nào **tái sử dụng** tổng của đoạn trước không?

In [1]:
nums = [1, 12, -5, -6, 50, 3]
k = 4
print("n =", len(nums), "| số đoạn dài k =", len(nums) - k + 1)
for i in range(len(nums) - k + 1):
    window = nums[i:i + k]
    print(f"i={i}: {window} tổng={sum(window)} trung bình={sum(window) / k}")

n = 6 | số đoạn dài k = 3
i=0: [1, 12, -5, -6] tổng=2 trung bình=0.5
i=1: [12, -5, -6, 50] tổng=51 trung bình=12.75
i=2: [-5, -6, 50, 3] tổng=42 trung bình=10.5


## 2. Nền tảng Python cần biết

### 2.1. Slicing `nums[i:j]` — cắt một đoạn liên tiếp

`nums[i:j]` trả về **list mới** gồm các phần tử từ chỉ số `i` đến **trước** `j` (không lấy `j`).
Vì vậy đoạn dài `k` bắt đầu tại `i` là `nums[i:i+k]`.

Lưu ý: slicing **tạo bản sao** → tốn `O(k)` thời gian và bộ nhớ mỗi lần gọi.

In [2]:
a = [10, 20, 30, 40, 50]
print(a[0:2])   # [10, 20]  -> chỉ số 0, 1
print(a[1:4])   # [20, 30, 40]
print(a[3:5])   # [40, 50]
print(a[3:99])  # [40, 50]  -> vượt quá độ dài không lỗi, chỉ cắt tới cuối
print(len(a[1:1 + 3]), "phần tử khi cắt a[1:1+3]")

[10, 20]
[20, 30, 40]
[40, 50]
[40, 50]
3 phần tử khi cắt a[1:1+3]


### 2.2. `range(n - k + 1)` — vì sao lại `+ 1`?

Vị trí bắt đầu cuối cùng hợp lệ là `n - k` (để đoạn kết thúc đúng ở `n - 1`).
`range(stop)` chạy từ `0` đến `stop - 1`, nên muốn có cả `n - k` thì `stop = n - k + 1`.

Ví dụ `n = 6, k = 4`: các vị trí bắt đầu là `0, 1, 2` → `range(3)`.

In [3]:
n, k = 6, 4
print(list(range(n - k + 1)))   # [0, 1, 2]
print(list(range(n - k)))       # [0, 1]  -> thiếu đoạn cuối, lỗi off-by-one hay gặp

[0, 1, 2]
[0, 1]


### 2.3. Phép chia `/` trả về `float`, `//` chia lấy phần nguyên

Đề yêu cầu trả về **trung bình thực** (`float`). Trong Python 3:
- `51 / 4 = 12.75` (float) — đúng thứ ta cần.
- `51 // 4 = 12` — mất phần thập phân, **sai** cho bài này.
- Với số âm: `-7 // 2 = -4` (làm tròn xuống), còn `-7 / 2 = -3.5`.

In [4]:
print(51 / 4, type(51 / 4))
print(51 // 4, type(51 // 4))
print(-7 / 2, -7 // 2)

12.75 <class 'float'>
12 <class 'int'>
-3.5 -4


### 2.4. `max()`, `float('-inf')` và hàm `sum()`

- `sum(list)` cộng tất cả phần tử (`O(độ dài)`).
- `float('-inf')` là "âm vô cùng" — giá trị khởi tạo an toàn khi đi tìm **max**, vì mọi số đều lớn hơn nó.
- `max(a, b)` trả về số lớn hơn.

Vì `nums[i]` có thể **âm**, khởi tạo `best = 0` là **sai** (ví dụ mọi đoạn đều có tổng âm).

In [5]:
best = float('-inf')
for s in [-10, -3, -7]:
    best = max(best, s)
print("best =", best)          # -3, đúng

best_wrong = 0
for s in [-10, -3, -7]:
    best_wrong = max(best_wrong, s)
print("best nếu khởi tạo 0 =", best_wrong, "(sai!)")
print(sum([12, -5, -6, 50]))

best = -3
best nếu khởi tạo 0 = 0 (sai!)
51


## 3. Cách trực giác — Brute Force

Người mới thường nghĩ: "thử mọi đoạn dài `k`, tính tổng, giữ tổng lớn nhất".

```text
với mỗi i từ 0 đến n-k:
    s = tổng nums[i..i+k-1]
    best = max(best, s)
trả về best / k
```

**Mẹo nhỏ đã dùng ngay ở đây:** vì mọi đoạn có **cùng độ dài k**, đoạn có tổng lớn nhất cũng là đoạn có trung bình lớn nhất. Ta chỉ chia cho `k` **một lần** ở cuối.

In [6]:
def max_average_bruteforce(nums: list[int], k: int) -> float:
    best = float('-inf')
    for i in range(len(nums) - k + 1):
        s = sum(nums[i:i + k])      # O(k) cho mỗi i
        best = max(best, s)
    return best / k

print(max_average_bruteforce([1, 12, -5, -6, 50, 3], 4))  # 12.75
print(max_average_bruteforce([5], 1))                     # 5.0
print(max_average_bruteforce([-1, -2, -3], 2))            # -1.5

12.75
5.0
-1.5


### 3.1. Đếm chi phí của brute force

Số đoạn là `n - k + 1`, mỗi đoạn tốn `k` phép cộng → khoảng `(n - k + 1)·k` phép cộng.
Tệ nhất khi `k ≈ n/2`: khoảng `n²/4`. Với `n = 10^5` đó là **2.5 tỷ** phép cộng → quá chậm (Time Limit Exceeded).

In [7]:
def count_additions(n: int, k: int) -> int:
    return (n - k + 1) * k

for n, k in [(6, 4), (1000, 500), (100_000, 50_000)]:
    print(f"n={n:>6}, k={k:>6} -> khoảng {count_additions(n, k):,} phép cộng")

n=     6, k=     4 -> khoảng 12 phép cộng
n=  1000, k=   500 -> khoảng 250,500 phép cộng
n=100000, k= 50000 -> khoảng 2,500,050,000 phép cộng


## 4. Nút thắt và insight — hai cửa sổ kề nhau chung gần hết phần tử

So sánh đoạn bắt đầu tại `i = 0` và `i = 1` với `k = 4`:

```text
i=0:  [ 1, 12, -5, -6]
i=1:      [12, -5, -6, 50]
           \_________/  phần chung 3 phần tử
```

Tổng mới = tổng cũ **− phần tử rời khỏi cửa sổ** (`nums[0] = 1`) **+ phần tử vào cửa sổ** (`nums[4] = 50`):

```text
51 = 2 - 1 + 50
```

Công thức tổng quát khi cửa sổ kết thúc ở chỉ số `i` (với `i >= k`):

```python
window_sum = window_sum + nums[i] - nums[i - k]
```

Mỗi bước chỉ còn **2 phép tính** thay vì `k` phép cộng. Đó là **sliding window cố định**.

In [8]:
nums = [1, 12, -5, -6, 50, 3]
k = 4
s0 = sum(nums[0:4])
s1 = s0 - nums[0] + nums[4]
print("tổng i=0:", s0)
print("tổng i=1 tính lại:", sum(nums[1:5]), "| tính bằng trượt:", s1)
s2 = s1 - nums[1] + nums[5]
print("tổng i=2 tính lại:", sum(nums[2:6]), "| tính bằng trượt:", s2)

tổng i=0: 2
tổng i=1 tính lại: 51 | tính bằng trượt: 51
tổng i=2 tính lại: 42 | tính bằng trượt: 42


### 4.1. Chỉ số `i - k` là phần tử nào?

Khi phần tử mới vào là `nums[i]`, cửa sổ mới là `nums[i-k+1 .. i]`.
Cửa sổ cũ là `nums[i-k .. i-1]`. Phần tử **duy nhất** có trong cũ mà không có trong mới là `nums[i-k]`.

| `i` (vào) | `i - k` (ra) | Cửa sổ mới |
|---:|---:|---|
| 4 | 0 | chỉ số 1..4 |
| 5 | 1 | chỉ số 2..5 |

In [9]:
n, k = 6, 4
for i in range(k, n):
    print(f"vào nums[{i}], ra nums[{i - k}], cửa sổ mới = chỉ số {i - k + 1}..{i}")

vào nums[4], ra nums[0], cửa sổ mới = chỉ số 1..4
vào nums[5], ra nums[1], cửa sổ mới = chỉ số 2..5


## 5. Lời giải chính — Sliding Window, giảng từng đoạn

**Đoạn 1 — khởi tạo cửa sổ đầu tiên:** tính tổng `k` phần tử đầu.

```python
window_sum = sum(nums[:k])
best = window_sum
```

**Đoạn 2 — trượt cửa sổ:** với mỗi `i` từ `k` đến `n-1`, thêm `nums[i]`, bỏ `nums[i-k]`, cập nhật `best`.

```python
for i in range(k, len(nums)):
    window_sum += nums[i] - nums[i - k]
    best = max(best, window_sum)
```

**Đoạn 3 — chia một lần:** `return best / k`.

Lưu ý `best = window_sum` (không phải `0` hay `-inf`) vì cửa sổ đầu tiên đã là một ứng viên hợp lệ.

In [10]:
def max_average_sliding(nums: list[int], k: int) -> float:
    window_sum = sum(nums[:k])          # đoạn 1: cửa sổ đầu
    best = window_sum
    for i in range(k, len(nums)):       # đoạn 2: trượt
        window_sum += nums[i] - nums[i - k]
        best = max(best, window_sum)
    return best / k                     # đoạn 3: chia 1 lần

print(max_average_sliding([1, 12, -5, -6, 50, 3], 4))
print(max_average_sliding([5], 1))
print(max_average_sliding([0, 4, 0, 3, 2], 1))

12.75
5.0
4.0


### 5.1. Khi `k == n` thì sao?

`range(k, n)` = `range(n, n)` là **rỗng** → vòng lặp không chạy, trả về `sum(nums) / n`. Đúng, vì chỉ có một đoạn duy nhất.

In [11]:
print(list(range(3, 3)))
print(max_average_sliding([4, 0, 2], 3), "==", sum([4, 0, 2]) / 3)

[]
2.0 == 2.0


## 6. TRACE từng bước

Ví dụ `nums = [1, 12, -5, -6, 50, 3]`, `k = 4`.

| Bước | `i` vào | `nums[i]` | `i-k` ra | `nums[i-k]` | `window_sum` trước → sau | `best` |
|---:|---:|---:|---:|---:|---|---:|
| khởi tạo | – | – | – | – | `sum([1,12,-5,-6]) = 2` | 2 |
| 1 | 4 | 50 | 0 | 1 | 2 → 2 + 50 − 1 = **51** | 51 |
| 2 | 5 | 3 | 1 | 12 | 51 → 51 + 3 − 12 = **42** | 51 |

Kết quả: `51 / 4 = 12.75`.

In [12]:
def trace_max_average(nums: list[int], k: int) -> float:
    window_sum = sum(nums[:k])
    best = window_sum
    print(f"khởi tạo: cửa sổ {nums[:k]} -> window_sum={window_sum}, best={best}")
    for i in range(k, len(nums)):
        before = window_sum
        window_sum += nums[i] - nums[i - k]
        best = max(best, window_sum)
        print(f"i={i}: +nums[{i}]={nums[i]:>3}, -nums[{i-k}]={nums[i-k]:>3} | "
              f"{before} -> {window_sum} | cửa sổ {nums[i-k+1:i+1]} | best={best}")
    print("kết quả =", best, "/", k, "=", best / k)
    return best / k

trace_max_average([1, 12, -5, -6, 50, 3], 4)

khởi tạo: cửa sổ [1, 12, -5, -6] -> window_sum=2, best=2
i=4: +nums[4]= 50, -nums[0]=  1 | 2 -> 51 | cửa sổ [12, -5, -6, 50] | best=51
i=5: +nums[5]=  3, -nums[1]= 12 | 51 -> 42 | cửa sổ [-5, -6, 50, 3] | best=51
kết quả = 51 / 4 = 12.75


12.75

### 6.1. Trace một ví dụ toàn số âm

Đây là ca dễ sai nếu khởi tạo `best = 0`. Hãy quan sát `best` luôn âm.

In [13]:
trace_max_average([-5, -1, -8, -2, -9], 2)

khởi tạo: cửa sổ [-5, -1] -> window_sum=-6, best=-6
i=2: +nums[2]= -8, -nums[0]= -5 | -6 -> -9 | cửa sổ [-1, -8] | best=-6
i=3: +nums[3]= -2, -nums[1]= -1 | -9 -> -10 | cửa sổ [-8, -2] | best=-6
i=4: +nums[4]= -9, -nums[2]= -8 | -10 -> -11 | cửa sổ [-2, -9] | best=-6
kết quả = -6 / 2 = -3.0


-3.0

## 7. Code nộp LeetCode

### Cách chính — Sliding Window (khuyên dùng)

In [14]:
class Solution:
    def findMaxAverage(self, nums: list[int], k: int) -> float:
        window_sum = sum(nums[:k])
        best = window_sum
        for i in range(k, len(nums)):
            window_sum += nums[i] - nums[i - k]
            if window_sum > best:
                best = window_sum
        return best / k

sol = Solution()
print(sol.findMaxAverage([1, 12, -5, -6, 50, 3], 4))  # 12.75
print(sol.findMaxAverage([5], 1))                     # 5.0

12.75
5.0


### 7.1. Cách thứ hai — Prefix Sum (tổng tiền tố)

**Ý tưởng:** xây mảng `prefix` dài `n + 1` với `prefix[j] = nums[0] + ... + nums[j-1]` (và `prefix[0] = 0`).
Khi đó tổng đoạn `nums[i .. i+k-1]` bằng `prefix[i + k] - prefix[i]`.

Ví dụ `nums = [1, 12, -5, -6, 50, 3]` → `prefix = [0, 1, 13, 8, 2, 52, 55]`.
Tổng đoạn `i=1, k=4` = `prefix[5] - prefix[1] = 52 - 1 = 51`. ✔

Cách này dùng thêm `O(n)` bộ nhớ nhưng rất hữu ích khi phải trả lời **nhiều** câu hỏi tổng đoạn.
`itertools.accumulate(nums, initial=0)` dựng prefix nhanh (Python 3.8+).

In [15]:
from itertools import accumulate

nums = [1, 12, -5, -6, 50, 3]
prefix = list(accumulate(nums, initial=0))
print("prefix =", prefix)
print("tổng đoạn i=1,k=4 =", prefix[5] - prefix[1])

def max_average_prefix(nums: list[int], k: int) -> float:
    prefix = list(accumulate(nums, initial=0))
    best = max(prefix[i + k] - prefix[i] for i in range(len(nums) - k + 1))
    return best / k

print(max_average_prefix(nums, 4))

prefix = [0, 1, 13, 8, 2, 52, 55]
tổng đoạn i=1,k=4 = 51
12.75


In [16]:
from itertools import accumulate

class SolutionPrefix:
    def findMaxAverage(self, nums: list[int], k: int) -> float:
        prefix = list(accumulate(nums, initial=0))
        best = max(prefix[i + k] - prefix[i] for i in range(len(nums) - k + 1))
        return best / k

print(SolutionPrefix().findMaxAverage([1, 12, -5, -6, 50, 3], 4))

12.75


### 7.2. Cách thứ ba — Sliding window "gọn kiểu Python" với `zip`

`zip(nums[k:], nums)` ghép cặp (phần tử vào, phần tử ra): cặp thứ `j` là `(nums[k + j], nums[j])`.
Đây chính là `(nums[i], nums[i - k])` với `i = k + j`. Cách viết này ngắn nhưng tạo thêm list `nums[k:]` (`O(n)` bộ nhớ).

In [17]:
nums = [1, 12, -5, -6, 50, 3]
k = 4
print(list(zip(nums[k:], nums)))   # [(50, 1), (3, 12)]

def max_average_zip(nums: list[int], k: int) -> float:
    s = best = sum(nums[:k])
    for add, rem in zip(nums[k:], nums):
        s += add - rem
        best = max(best, s)
    return best / k

print(max_average_zip(nums, k))

[(50, 1), (3, 12)]
12.75


## 8. Vì sao Sliding Window đúng?

**Bất biến (invariant):** ngay sau khi xử lý chỉ số `i` (với `i >= k - 1`):
1. `window_sum` = tổng của `nums[i-k+1 .. i]` (đúng `k` phần tử kết thúc tại `i`).
2. `best` = tổng lớn nhất trong mọi cửa sổ kết thúc tại chỉ số `≤ i`.

- **Khởi tạo** (`i = k-1`): `window_sum = sum(nums[:k])`, `best = window_sum` → đúng cả hai.
- **Duy trì:** từ cửa sổ `[i-k .. i-1]` sang `[i-k+1 .. i]`, ta cộng đúng phần tử mới vào và trừ đúng phần tử bị loại → (1) giữ đúng; `best = max(best, window_sum)` → (2) giữ đúng.
- **Kết thúc** (`i = n-1`): `best` là tổng lớn nhất trong **mọi** `n-k+1` cửa sổ.

Cuối cùng vì mọi cửa sổ có cùng độ dài `k > 0`, `tổng lớn nhất / k` = `trung bình lớn nhất` (chia cho số dương giữ nguyên thứ tự).

## 9. Độ phức tạp và bảng so sánh

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time | Space thêm | Ghi chú |
|---|---|---:|---:|---|
| Brute force | `sum(nums[i:i+k])` cho mọi `i` | `O(n·k)` | `O(k)` (slice) | TLE khi `n = 10^5` |
| Prefix sum | `prefix[i+k] - prefix[i]` | `O(n)` | `O(n)` | Mạnh khi nhiều truy vấn |
| Sliding window | `+nums[i] - nums[i-k]` | `O(n)` | `O(1)` | **Tối ưu, nên dùng** |
| Sliding + `zip` | như trên, viết gọn | `O(n)` | `O(n)` (`nums[k:]`) | Pythonic |

Sliding window: `sum(nums[:k])` tốn `O(k)`, vòng lặp chạy `n - k` lần → tổng `O(n)`.

In [18]:
import time, random

rng = random.Random(643)
big = [rng.randint(-10**4, 10**4) for _ in range(20000)]
for name, f in [("sliding", max_average_sliding), ("prefix", max_average_prefix), ("brute", max_average_bruteforce)]:
    t = time.perf_counter()
    r = f(big, 2000)
    print(f"{name:>8}: {r:.5f}  ({time.perf_counter() - t:.3f}s)")

 sliding: 304.21700  (0.003s)
  prefix: 304.21700  (0.003s)


   brute: 304.21700  (0.309s)


## 10. Bộ test

Nhóm test:
1. Ví dụ đề bài.
2. Biên: `n = 1`, `k = 1`, `k = n`.
3. Toàn số âm (bẫy khởi tạo `0`).
4. Cửa sổ tốt nhất ở **đầu** / **cuối** mảng.
5. Đối chiếu ngẫu nhiên giữa các cách.

Vì kết quả là `float`, ta so sánh bằng `abs(a - b) < 1e-5` như đề cho phép.

In [19]:
def close(a: float, b: float) -> bool:
    return abs(a - b) < 1e-5

cases = [
    ([1, 12, -5, -6, 50, 3], 4, 12.75),
    ([5], 1, 5.0),
    ([4, 0, 2], 3, 2.0),                 # k == n
    ([-5, -1, -8, -2, -9], 2, -3.0),     # toàn âm: các tổng -6,-9,-10,-11 -> max -6
    ([9, 9, 1, 1, 1], 2, 9.0),           # tốt nhất ở đầu
    ([1, 1, 1, 9, 9], 2, 9.0),           # tốt nhất ở cuối
    ([0, 1, 1, 3, 3], 4, 2.0),
]

sol = Solution()
for nums, k, expected in cases:
    for f in (sol.findMaxAverage, SolutionPrefix().findMaxAverage, max_average_bruteforce, max_average_zip):
        got = f(nums, k)
        assert close(got, expected), (f.__name__, nums, k, got, expected)
    print(f"{str(nums):<24} k={k} -> {expected} PASS")

[1, 12, -5, -6, 50, 3]   k=4 -> 12.75 PASS
[5]                      k=1 -> 5.0 PASS
[4, 0, 2]                k=3 -> 2.0 PASS
[-5, -1, -8, -2, -9]     k=2 -> -3.0 PASS
[9, 9, 1, 1, 1]          k=2 -> 9.0 PASS
[1, 1, 1, 9, 9]          k=2 -> 9.0 PASS
[0, 1, 1, 3, 3]          k=4 -> 2.0 PASS


### 10.1. Test đối chiếu ngẫu nhiên (có seed)

So sliding window với brute force trên 500 mảng ngẫu nhiên nhỏ.

In [20]:
rng = random.Random(2024)
for _ in range(500):
    n = rng.randint(1, 12)
    nums = [rng.randint(-20, 20) for _ in range(n)]
    k = rng.randint(1, n)
    a = Solution().findMaxAverage(nums, k)
    b_ = max_average_bruteforce(nums, k)
    c = max_average_prefix(nums, k)
    assert close(a, b_) and close(a, c), (nums, k, a, b_, c)
print("500 test ngẫu nhiên: sliding == brute == prefix  PASS")

500 test ngẫu nhiên: sliding == brute == prefix  PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Khởi tạo `best = 0`
Mảng toàn số âm → `best` giữ mãi `0` → trả về `0.0` sai.
**Sửa:** `best = window_sum` (cửa sổ đầu tiên) hoặc `float('-inf')`.

In [21]:
def wrong_init_zero(nums, k):
    window_sum = sum(nums[:k])
    best = 0                                # SAI
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]
        best = max(best, window_sum)
    return best / k

print("sai:", wrong_init_zero([-5, -1, -8], 2), "| đúng:", Solution().findMaxAverage([-5, -1, -8], 2))

sai: 0.0 | đúng: -3.0


### Lỗi 2: Quên so sánh cửa sổ đầu tiên
Nếu khởi tạo `best = float('-inf')` nhưng chỉ cập nhật `best` **trong vòng lặp**, cửa sổ đầu không bao giờ được xét. Khi `k == n` vòng lặp rỗng → trả về `-inf`!

In [22]:
def wrong_forget_first(nums, k):
    window_sum = sum(nums[:k])
    best = float('-inf')                    # SAI: chưa so window đầu
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]
        best = max(best, window_sum)
    return best / k

print("sai:", wrong_forget_first([4, 0, 2], 3), "| đúng:", Solution().findMaxAverage([4, 0, 2], 3))
print("sai:", wrong_forget_first([9, 1, 1], 2), "| đúng:", Solution().findMaxAverage([9, 1, 1], 2))

sai: -inf | đúng: 2.0
sai: 1.0 | đúng: 5.0


### Lỗi 3: Trừ nhầm phần tử `nums[i - k + 1]`
Phần tử rời cửa sổ là `nums[i - k]`. Trừ `nums[i-k+1]` sẽ bỏ nhầm phần tử vẫn còn trong cửa sổ mới.

### Lỗi 4: Dùng `//` thay vì `/`
`51 // 4 = 12` chứ không phải `12.75`.

### Lỗi 5: Chia trong mỗi bước rồi so sánh float
Không sai về logic nhưng dư thừa phép chia và dễ gây sai số tích lũy nếu cập nhật trung bình kiểu `avg += (add - rem) / k`. Giữ **tổng nguyên**, chia một lần ở cuối.

In [23]:
def wrong_index(nums, k):
    s = sum(nums[:k]); best = s
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k + 1]      # SAI chỉ số
        best = max(best, s)
    return best / k

print("sai chỉ số:", wrong_index([1, 12, -5, -6, 50, 3], 4), "| đúng:", 12.75)
print("dùng //:", 51 // 4, "| dùng /:", 51 / 4)

sai chỉ số: 12.0 | đúng: 12.75
dùng //: 12 | dùng /: 12.75


## 12. Mẹo nhận dạng pattern

Dấu hiệu dùng **sliding window độ dài cố định**:
- Đề nói "subarray / substring **liên tiếp** có độ dài **k**".
- Cần max/min/đếm của một đại lượng **cộng dồn được** (tổng, số nguyên âm, số ký tự thỏa điều kiện...).
- Khi cửa sổ dịch 1 bước, đại lượng có thể **cập nhật O(1)** bằng "thêm phần tử vào, bỏ phần tử ra".

Template:
```python
cur = f(nums[:k])
best = cur
for i in range(k, n):
    cur = cur + g(nums[i]) - g(nums[i - k])
    best = max(best, cur)
```

Ngược lại, nếu độ dài **không cố định** (ví dụ "đoạn ngắn nhất có tổng ≥ target") → sliding window **co giãn** với hai con trỏ.

## 13. Biến thể

### 13.1. Trả về **vị trí bắt đầu** của đoạn tốt nhất
Chỉ cần nhớ thêm `best_start` mỗi khi `best` được cập nhật.

In [24]:
def best_window_start(nums: list[int], k: int) -> int:
    s = best = sum(nums[:k])
    best_start = 0
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        if s > best:
            best, best_start = s, i - k + 1
    return best_start

st = best_window_start([1, 12, -5, -6, 50, 3], 4)
print("bắt đầu tại", st, "->", [1, 12, -5, -6, 50, 3][st:st + 4])

bắt đầu tại 1 -> [12, -5, -6, 50]


### 13.2. Đếm số đoạn dài `k` có trung bình ≥ `threshold` (LeetCode 1343)
Trung bình ≥ threshold ⇔ tổng ≥ `threshold * k` → vẫn so sánh tổng, không cần chia.

In [25]:
def count_windows_at_least(nums: list[int], k: int, threshold: int) -> int:
    need = threshold * k
    s = sum(nums[:k])
    count = 1 if s >= need else 0
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        if s >= need:
            count += 1
    return count

print(count_windows_at_least([2, 2, 2, 2, 5, 5, 5, 8], 3, 4))  # 3

3


### 13.3. Maximum Average Subarray **II** (LeetCode 644 — khó)
Độ dài **≥ k** thay vì đúng `k`. Không trượt cố định được nữa; lời giải chuẩn dùng **binary search trên đáp án** + prefix sum. Ở đây chỉ cần biết: "độ dài thay đổi" là dấu hiệu bài khó hơn hẳn.

## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** với `nums = [3, -1, 4, 1, -5, 9]`, `k = 3`, liệt kê tổng các cửa sổ theo cách trượt (ghi rõ phần tử vào/ra) và cho biết trung bình lớn nhất.

**Bài 2 — Đoán output:** cho đoạn code dưới, dự đoán in ra gì.
```python
nums = [2, 7, 1, 8]; k = 2
s = sum(nums[:k])
for i in range(k, len(nums)):
    s += nums[i] - nums[i - k]
    print(i, s)
```

**Bài 3 — TODO:** hoàn thiện `exercise_max_sum_k` trả về **tổng** lớn nhất của đoạn dài `k`.

In [26]:
def exercise_max_sum_k(nums: list[int], k: int) -> int:
    # TODO 1: tính tổng cửa sổ đầu tiên
    # TODO 2: trượt từ i = k đến n-1, thêm nums[i], bỏ nums[i-k]
    # TODO 3: cập nhật best và trả về
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_sum_k([1, 12, -5, -6, 50, 3], 4) == 51
# assert exercise_max_sum_k([-3, -2, -9], 1) == -2

**Bài 4 — TODO biến thể:** đếm số đoạn dài `k` có tổng **chia hết cho `m`**.

In [27]:
def exercise_count_divisible(nums: list[int], k: int, m: int) -> int:
    # TODO: sliding window, kiểm tra s % m == 0 cho mỗi cửa sổ (kể cả cửa sổ đầu)
    pass

# assert exercise_count_divisible([1, 2, 3, 4, 5], 2, 3) == 2   # [1,2]=3, [4,5]=9

**Bài 5 — TODO chuỗi:** cho chuỗi `s` và `k`, trả về **số nguyên âm lớn nhất** trong một đoạn con dài `k` (LeetCode 1456). Nguyên âm: `a e i o u`.

In [28]:
def exercise_max_vowels(s: str, k: int) -> int:
    # TODO: đếm nguyên âm của cửa sổ đầu, rồi trượt: +1 nếu ký tự vào là nguyên âm, -1 nếu ký tự ra là nguyên âm
    pass

# assert exercise_max_vowels("abciiidef", 3) == 3
# assert exercise_max_vowels("leetcode", 3) == 2

**Bài 6 — Mini challenge:** trả về **tất cả** vị trí bắt đầu của đoạn dài `k` đạt tổng lớn nhất (có thể có nhiều đoạn hòa nhau).

In [29]:
def exercise_all_best_starts(nums: list[int], k: int) -> list[int]:
    # TODO: khi s > best -> reset danh sách; khi s == best -> thêm vào
    pass

# assert exercise_all_best_starts([1, 3, 1, 3, 1], 2) == [0, 1, 2, 3]
# assert exercise_all_best_starts([5, 0, 0, 5], 1) == [0, 3]

## 15. Gợi ý 3 mức

| Bài | Mức 1 (hướng nghĩ) | Mức 2 (cấu trúc) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | Bắt đầu bằng tổng 3 phần tử đầu | Mỗi bước: vào `nums[i]`, ra `nums[i-3]` | 6 → 4 → 0 → 5 |
| 2 | Chỉ in khi `i = 2, 3` | `s` ban đầu = 9 | `9 - 2 + 1`, rồi `- 7 + 8` |
| 3 | Giống lời giải chính, bỏ phép chia | `s = best = sum(nums[:k])` | `s += nums[i] - nums[i-k]` |
| 4 | Đếm cả cửa sổ đầu tiên | `count = 1 if s % m == 0 else 0` | Python `%` với số âm vẫn trả 0 khi chia hết |
| 5 | Biến mỗi ký tự thành 0/1 | `is_v = lambda c: c in "aeiou"` | `cur += is_v(s[i]) - is_v(s[i-k])` |
| 6 | Cần nhớ list chỉ số | `if s > best: best, starts = s, [start]` | `elif s == best: starts.append(start)` |

## 16. Đáp án tham khảo

**Bài 1:** cửa sổ đầu `[3,-1,4] = 6`; vào 1 ra 3 → 4; vào −5 ra −1 → 0; vào 9 ra 4 → 5. Max = 6 → trung bình `2.0`.

**Bài 2:** `s` ban đầu `= 2 + 7 = 9`. Với `i = 2`: `9 + 1 − 2 = 8` → in `2 8`. Với `i = 3`: `8 + 8 − 7 = 9` → in `3 9`.

Cell dưới chạy lại hai bài để đối chiếu.

In [30]:
# Kiểm tra Bài 1 và Bài 2
nums = [3, -1, 4, 1, -5, 9]; k = 3
print("Bài 1:", Solution().findMaxAverage(nums, k))

nums = [2, 7, 1, 8]; k = 2
s = sum(nums[:k])
for i in range(k, len(nums)):
    s += nums[i] - nums[i - k]
    print(i, s)

Bài 1: 2.0
2 8
3 9


In [31]:
def answer_max_sum_k(nums: list[int], k: int) -> int:
    s = best = sum(nums[:k])
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        best = max(best, s)
    return best


def answer_count_divisible(nums: list[int], k: int, m: int) -> int:
    s = sum(nums[:k])
    count = 1 if s % m == 0 else 0
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        if s % m == 0:
            count += 1
    return count


def answer_max_vowels(s: str, k: int) -> int:
    vowels = set("aeiou")
    cur = sum(1 for c in s[:k] if c in vowels)
    best = cur
    for i in range(k, len(s)):
        cur += (s[i] in vowels) - (s[i - k] in vowels)
        best = max(best, cur)
    return best


def answer_all_best_starts(nums: list[int], k: int) -> list[int]:
    s = best = sum(nums[:k])
    starts = [0]
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        start = i - k + 1
        if s > best:
            best, starts = s, [start]
        elif s == best:
            starts.append(start)
    return starts


assert answer_max_sum_k([1, 12, -5, -6, 50, 3], 4) == 51
assert answer_max_sum_k([-3, -2, -9], 1) == -2
assert answer_count_divisible([1, 2, 3, 4, 5], 2, 3) == 2
assert answer_max_vowels("abciiidef", 3) == 3
assert answer_max_vowels("leetcode", 3) == 2
assert answer_all_best_starts([1, 3, 1, 3, 1], 2) == [0, 1, 2, 3]
assert answer_all_best_starts([5, 0, 0, 5], 1) == [0, 3]
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findMaxAverage(self, nums: list[int], k: int) -> float:
        window_sum = sum(nums[:k])          # cửa sổ đầu
        best = window_sum
        for i in range(k, len(nums)):       # trượt
            window_sum += nums[i] - nums[i - k]
            best = max(best, window_sum)
        return best / k                     # chia 1 lần
```

**Câu thần chú:** *"Vào `nums[i]`, ra `nums[i-k]`; so tổng, chia cuối."*

**Checklist trước khi submit:**
- [ ] `best` khởi tạo bằng tổng cửa sổ đầu (không phải `0`).
- [ ] Vòng lặp bắt đầu từ `i = k`, trừ đúng `nums[i - k]`.
- [ ] Dùng `/` (float), không dùng `//`.
- [ ] Đã nghĩ tới `k == n` (vòng lặp rỗng) và mảng toàn số âm.
- [ ] Độ phức tạp `O(n)` time, `O(1)` space.